# Normalization

In [1]:
import os
import json
from pathlib import Path
from typing import Any

from travelplanner.utils.eval import print_scorecard

## Constants

In [2]:
TP_EVAL_PATH = Path("../data/evaluation/travel_agent/")
BL_EVAL_PATH = Path("../data/evaluation/baseline/")

TP_TRACES_PATH = Path("../data/traces/travel_agent_full/")
BL_TRACES_PATH = Path("../data/traces/baseline/")


## First Load

In [3]:
all_travel_queries = json.loads(Path("../data/travel_queries_all_30.json").read_text())
query_ids = [
  ("_".join(path.as_posix().split("/")[-1].split("_")[:2]), path.resolve().as_posix())
  for path in TP_EVAL_PATH.iterdir()
  if path.is_dir()
]
query_ids.sort(key=lambda x: int(x[0][6:]))
print(query_ids)

[('query_1', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_1_couple_citytrip_Adrian'), ('query_2', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_2_family_Adrian'), ('query_3', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_3_no_flight_overseas_Adrian'), ('query_4', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_4_special_interest_astronomy_Adrian'), ('query_5', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_5_special_interest_vegan_Adrian'), ('query_6', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_6_tight_budget_Adrian'), ('query_7', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_7_business_solo_Adrian'), ('query_8', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_8_friends_group_tokyo_Adrian'), ('query_9', '/home/justus/ie686_llm_project/data/evaluation/travel_agent/query_9_honeymoon_maldives_Adrian'), ('query_10', '/home/ju

In [4]:
def get_travelquery(query_id: str) -> dict:
    for query in all_travel_queries:
        if query["id"].startswith(query_id):
            return query
    raise ValueError(f"No query found for id: '{query_id}'")

In [5]:
query_dict: dict[str, dict[str, Path|str|object]] = {}

for id, path in query_ids:
  travelquery = get_travelquery(id)
  tp_eval_json = json.loads(Path(path+".json").read_text())
  bl_eval_json = json.loads(Path(path.replace("travel_agent", "baseline")+".json").read_text())
  tp_plan_path = Path("..") / tp_eval_json["plan_path"]
  bl_plan_path = Path("..") / bl_eval_json["plan_path"]
  query_dict[id] = {
    "query": travelquery["query"],
    "type": travelquery["type"],
    "hard_constraints": travelquery["hard_constraints"],
    "tp_eval_path": Path(path),
    "tp_scorecard": tp_eval_json["scorecard"],
    "tp_plan_path": tp_plan_path,
    "tp_output": json.loads(tp_plan_path.read_text()),
    "bl_eval_path": Path(path.replace("travel_agent", "baseline")),
    "bl_scorecard": bl_eval_json["scorecard"],
    "bl_plan_path": bl_plan_path,
    "bl_output": bl_plan_path.read_text(),
  }
  

In [6]:
bl_scorecard = json.loads(
  (query_dict['query_8']['bl_eval_path'] / "scorecard.json").read_text()
)
tp_scorecard = json.loads(
  (query_dict['query_8']['tp_eval_path'] / "scorecard.json").read_text()
)


In [ ]:
print_scorecard(bl_scorecard, "BASELINE: query_8")
print_scorecard(tp_scorecard, "TRAVEL AGENT: query_8")

SCORECARD — BASELINE: query_1
Rationale verification : 57 PASS, 2 FAIL, 7 MISSING_INFO  (of 66 slots)
HC  Micro Pass Rate    : 87.5%
CC  Micro Pass Rate    : 76.9%
HC  Macro Pass Rate    : 0%
CC  Macro Pass Rate    : 0%
Final Pass Rate        : 0%

Rationale verification detail:
  [PASS        ] Day 1/slot 1 — Depart Munich for Tokyo  (source: web_search)
  [PASS        ] Day 1/slot 2 — Overnight flight / connection  (source: web_search)
  [PASS        ] Day 1/slot 3 — In flight meal  (source: web_search)
  [PASS        ] Day 2/slot 1 — Arrive in Tokyo and transfer  (source: web_search)
  [PASS        ] Day 2/slot 2 — Check in at CITAN  (source: link)
  [PASS        ] Day 2/slot 3 — Rest and group planning  (source: web_search)
  [PASS        ] Day 2/slot 4 — Asakusa and Senso-ji walk  (source: link)
  [PASS        ] Day 2/slot 5 — Street-food-style dinner  (source: web_search)
  [MISSING_INFO] Day 2/slot 6 — Low-key drinks  (source: web_search)
               reason: The provided evid

In [8]:
import re
from collections import Counter

import pandas as pd

CATEGORIES = ["meal", "attraction", "transport", "lodging", "leisure", "other"]


def _money_numbers(text: str) -> list[float]:
    return [float(value.replace(",", "")) for value in re.findall(r"€\s*([0-9][0-9,]*(?:\.[0-9]+)?)", text)]


def _scorecard_summary(scorecard: dict[str, Any]) -> dict[str, Any]:
    checks = (
        scorecard.get("aggregated_constraints")
        or scorecard.get("constraint_evaluations")
        or scorecard.get("constraint_checks")
        or []
    )
    verdicts = Counter(check.get("final_verdict", "UNKNOWN") for check in checks)
    hard_checks = [check for check in checks if check.get("constraint_type") == "hard"]
    hard_verdicts = Counter(check.get("final_verdict", "UNKNOWN") for check in hard_checks)
    total = sum(verdicts.values())
    hard_total = sum(hard_verdicts.values())
    rationale_total = (
        scorecard.get("rationale_pass_count", 0)
        + scorecard.get("rationale_fail_count", 0)
        + scorecard.get("rationale_missing_count", 0)
    )
    return {
        "hard_constraint_checks": hard_total,
        "hard_constraint_pass": hard_verdicts.get("PASS", 0),
        "hard_constraint_fail": hard_verdicts.get("FAIL", 0),
        "hard_constraint_missing": hard_verdicts.get("MISSING_INFO", 0),
        "hard_constraint_pass_rate": round(hard_verdicts.get("PASS", 0) / hard_total, 3) if hard_total else None,
        "hc_micro_pass_rate": scorecard.get("hc_micro_pass_rate"),
        "hc_macro_pass_rate": scorecard.get("hc_macro_pass_rate"),
        "rationale_checks": rationale_total,
        "rationale_pass": scorecard.get("rationale_pass_count", 0),
        "rationale_fail": scorecard.get("rationale_fail_count", 0),
        "rationale_missing": scorecard.get("rationale_missing_count", 0),
        "rationale_pass_rate": round(scorecard.get("rationale_pass_count", 0) / rationale_total, 3) if rationale_total else None,
        "scorecard_checks": total,
        "scorecard_pass": verdicts.get("PASS", 0),
        "scorecard_fail": verdicts.get("FAIL", 0),
        "scorecard_missing": verdicts.get("MISSING_INFO", 0),
        "scorecard_pass_rate": round(verdicts.get("PASS", 0) / total, 3) if total else None,
    }


def _field_presence(slots: list[dict[str, Any]], fields: list[str]) -> dict[str, float]:
    if not slots:
        return {field: 0.0 for field in fields}
    presence = {}
    for field in fields:
        present = sum(1 for slot in slots if slot.get(field) not in (None, "", [], {}))
        presence[field] = round(present / len(slots), 3)
    return presence


def _baseline_category(raw_type: str, plan_text: str) -> str:
    text = f"{raw_type} {plan_text}".lower()
    if any(word in text for word in ["food", "breakfast", "lunch", "dinner", "coffee", "meal"]):
        return "meal"
    if any(word in text for word in ["transport", "train", "transit", "transfer"]):
        return "transport"
    if any(word in text for word in ["accommodation", "hotel", "check in", "check-in", "check out", "check-out"]):
        return "lodging"
    if any(word in text for word in ["museum", "palace", "attraction"]):
        return "attraction"
    if any(word in text for word in ["walk", "stroll", "free time", "break"]):
        return "leisure"
    return "other"


def _parse_baseline_markdown(plan_md: str) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    slots = []
    current_day = None
    for line in plan_md.splitlines():
        day_match = re.match(r"## Day (\d+)", line)
        if day_match:
            current_day = int(day_match.group(1))
            continue
        if not current_day or not line.startswith("|") or "---" in line or " Time " in line:
            continue
        cells = [cell.strip() for cell in line.strip("|").split("|")]
        if len(cells) != 6:
            continue
        time, raw_type, plan, location, cost_text, evidence_notes = cells
        if not time or not plan:
            continue
        slots.append({
            "day": current_day,
            "time": time,
            "category": _baseline_category(raw_type, plan),
            "location": location,
            "cost": cost_text,
            "links/evidence": evidence_notes if "http" in evidence_notes or evidence_notes else "",
            "notes": evidence_notes,
        })
    budget_match = re.search(r"Estimated total:\s*\*\*about\s+€([0-9,]+)\s*[–-]\s*€([0-9,]+)", plan_md)
    if budget_match:
        low, high = [float(value.replace(",", "")) for value in budget_match.groups()]
        total_cost = round((low + high) / 2, 2)
        total_cost_display = f"€{low:,.0f}-€{high:,.0f} midpoint €{total_cost:,.2f}"
    else:
        costs = [numbers[-1] for slot in slots if (numbers := _money_numbers(slot["cost"]))]
        total_cost = round(sum(costs), 2)
        total_cost_display = f"€{total_cost:,.2f}"
    complete_final_answer = bool(slots) and "## Budget summary" in plan_md and len({slot["day"] for slot in slots}) >= 5
    return slots, {"total_cost": total_cost, "total_cost_display": total_cost_display, "complete_final_answer": complete_final_answer}


def _summarize_plan(system: str, query_id: str, query_info: dict[str, Any]) -> dict[str, Any]:
    eval_path = query_info[f"{system}_eval_path"]
    eval_json = json.loads(Path(f"{eval_path}.json").read_text())
    scorecard = json.loads((eval_path / "scorecard.json").read_text())
    plan_path = Path("..") / eval_json["plan_path"]

    if system == "tp":
        plan_json = json.loads(plan_path.read_text())
        days = plan_json["travelplan"].get("days", [])
        slots = [
            {
                "day": day.get("index"),
                "time": f"{slot.get('start_time', '')} - {slot.get('end_time', '')}",
                "category": slot.get("category", "other"),
                "location": slot.get("location"),
                "cost": slot.get("cost"),
                "links/evidence": slot.get("links"),
                "notes": slot.get("notes"),
            }
            for day in days
            for slot in day.get("slots", [])
        ]
        total_cost = round(sum(float(slot.get("cost") or 0) for day in days for slot in day.get("slots", [])), 2)
        status = f"passed={plan_json.get('validation_passed')} attempts={plan_json.get('validation_attempts')}"
        complete_final_answer = None
        total_cost_display = f"€{total_cost:,.2f}"
    else:
        slots, baseline_meta = _parse_baseline_markdown(plan_path.read_text())
        days = [{"index": day} for day in sorted({slot["day"] for slot in slots})]
        total_cost = baseline_meta["total_cost"]
        total_cost_display = baseline_meta["total_cost_display"]
        status = None
        complete_final_answer = baseline_meta["complete_final_answer"] and eval_json.get("status") == "ok"

    day_counts = Counter(slot["day"] for slot in slots)
    category_counts = Counter(slot.get("category") if slot.get("category") in CATEGORIES else "other" for slot in slots)
    required_fields = _field_presence(slots, ["time", "location", "cost", "links/evidence", "notes"])

    return {
        "query_id": query_id,
        "system": "Travel Agent" if system == "tp" else "Baseline",
        "number_of_days": len(days),
        "number_of_slots": len(slots),
        "slots_per_day": dict(sorted(day_counts.items())),
        **{f"category_{category}": category_counts.get(category, 0) for category in CATEGORIES},
        "total_estimated_cost": total_cost_display,
        "cost_per_day": f"€{(total_cost / len(days)):,.2f}" if days else None,
        **{f"field_presence_{field}": value for field, value in required_fields.items()},
        "travel_agent_validation_status": status,
        "baseline_complete_final_answer": complete_final_answer,
        **_scorecard_summary(scorecard),
    }


query_id = "query_1" if "query_1" in query_dict else "query1"
basic_structure_comparison = pd.DataFrame([
    _summarize_plan("bl", query_id, query_dict[query_id]),
    _summarize_plan("tp", query_id, query_dict[query_id]),
]).set_index(["query_id", "system"])

basic_structure_comparison.T


query_id                                                 query_1  \
system                                                  Baseline   
number_of_days                                                 5   
number_of_slots                                               33   
slots_per_day                     {1: 7, 2: 7, 3: 6, 4: 7, 5: 6}   
category_meal                                                 15   
category_attraction                                            6   
category_transport                                             3   
category_lodging                                               3   
category_leisure                                               5   
category_other                                                 1   
total_estimated_cost            €1,089-€1,239 midpoint €1,164.00   
cost_per_day                                             €232.80   
field_presence_time                                          1.0   
field_presence_location                                      1.0   
field_presence_cost                                          1.0   
field_presence_links/evidence                                1.0   
field_presence_notes                                         1.0   
travel_agent_validation_status                               NaN   
baseline_complete_final_answer                              True   
hard_constraint_checks                                         7   
hard_constraint_pass                                           6   
hard_constraint_fail                                           1   
hard_constraint_missing                                        0   
hard_constraint_pass_rate                                  0.857   
hc_micro_pass_rate                                        0.8571   
hc_macro_pass_rate                                           0.0   
rationale_checks                                              33   
rationale_pass                                                29   
rationale_fail                                                 0   
rationale_missing                                              4   
rationale_pass_rate                                        0.879   
scorecard_checks                                              20   
scorecard_pass                                                18   
scorecard_fail                                                 2   
scorecard_missing                                              0   
scorecard_pass_rate                                          0.9   

query_id                                                        
system                                            Travel Agent  
number_of_days                                               5  
number_of_slots                                             30  
slots_per_day                   {1: 7, 2: 6, 3: 6, 4: 6, 5: 5}  
category_meal                                               13  
category_attraction                                          3  
category_transport                                           8  
category_lodging                                             1  
category_leisure                                             5  
category_other                                               0  
total_estimated_cost                                 €1,191.40  
cost_per_day                                           €238.28  
field_presence_time                                        1.0  
field_presence_location                                    1.0  
field_presence_cost                                        1.0  
field_presence_links/evidence                            0.933  
field_presence_notes                                       1.0  
travel_agent_validation_status          passed=True attempts=2  
baseline_complete_final_answer                            None  
hard_constraint_checks                                       7  
hard_constraint_pass                                         7  
hard_constraint_fail                                 